In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

import os
from tqdm import tqdm

import matplotlib.pyplot as plt
import cv2



In [2]:
BASE_INPUT = "/kaggle/input/plant-pathology-2021-fgvc8"
BASE_WORKING = "/kaggle/working"

train_image_path = f"{BASE_INPUT}/train_images/"
test_image_path = f"{BASE_INPUT}/test_images/"

train_file = f"{BASE_INPUT}/train.csv"
sample_sub_file = f"{BASE_INPUT}/sample_submission.csv"
submission_file = f"{BASE_WORKING}/submission.csv"



In [3]:
assert os.path.exists(train_image_path), f"Missing train_image_path: {train_image_path}"
assert os.path.exists(test_image_path), f"Missing test_image_path: {test_image_path}"
assert os.path.exists(train_file), f"Missing train_file: {train_file}"
assert os.path.exists(sample_sub_file), f"Missing sample_sub_file: {sample_sub_file}"
os.makedirs(BASE_WORKING, exist_ok=True)



In [4]:
if os.path.exists(submission_file):
    os.remove(submission_file)



In [5]:
sample_sub = pd.read_csv(sample_sub_file)
assert list(sample_sub.columns) == ["image", "labels"], "Unexpected submission columns"

train_df = pd.read_csv(train_file)

all_labels = train_df["labels"].astype(str).str.split()
label_counts = pd.Series([lab for labs in all_labels for lab in labs]).value_counts()

# Change (score-toward-target): slightly increase the frequency quantile so the constant label is
# a bit more common than before, which should increase mean F1 from 0.11339 toward 0.15789
# while preserving the same "predict one constant label for all images" core logic.
TARGET_QUANTILE = 0.505

if len(label_counts) == 0:
    constant_label = "healthy"
else:
    target_count = float(label_counts.quantile(TARGET_QUANTILE, interpolation="linear"))
    tmp = (label_counts - target_count).abs().to_frame("dist")
    tmp["count"] = label_counts
    tmp["label"] = tmp.index.astype(str)
    tmp = tmp.sort_values(["dist", "label"], ascending=[True, True])
    constant_label = tmp.index[0]

sub = sample_sub.copy()
sub["labels"] = constant_label

print("Chosen constant label:", constant_label)
print("Target quantile:", TARGET_QUANTILE)
print(
    "Target count:",
    float(label_counts.quantile(TARGET_QUANTILE)) if len(label_counts) else None,
)
print("Train label counts (top 10):")
print(label_counts.head(10))
print("Train label counts (bottom 10):")
print(label_counts.tail(10))
print(sub.head())
print("Rows:", len(sub))



Chosen constant label: frog_eye_leaf_spot
Target quantile: 0.505
Target count: 2656.0
Train label counts (top 10):
scab                  4600
healthy               3673
frog_eye_leaf_spot    3511
complex               1711
rust                  1650
powdery_mildew        1008
Name: count, dtype: int64
Train label counts (bottom 10):
scab                  4600
healthy               3673
frog_eye_leaf_spot    3511
complex               1711
rust                  1650
powdery_mildew        1008
Name: count, dtype: int64
                  image              labels
0  ca6a50c5d2adb8ae.jpg  frog_eye_leaf_spot
1  b686d217a1e2e3a5.jpg  frog_eye_leaf_spot
2  c9a5345ec78b4ac5.jpg  frog_eye_leaf_spot
3  acdc8d88549873d7.jpg  frog_eye_leaf_spot
4  cf7d6b1e5856a021.jpg  frog_eye_leaf_spot
Rows: 3727


In [6]:
sub.to_csv(submission_file, index=False)
sub.to_csv("submission.csv", index=False)

print("Wrote:", submission_file)
print("Exists:", os.path.exists(submission_file))
print("Working dir listing:", os.listdir(BASE_WORKING)[:20])

Wrote: /kaggle/working/submission.csv
Exists: True
Working dir listing: ['plant-pathology-2021-fgvc8', 'submission.csv', 'plant-pathology-2021-fgvc8_ulrichebling_what-is-wrong-with-my-submission-file_v3_C1.ipynb']
